# Clase 16 — Evaluación Rigurosa de Clasificadores: Plotly, Train/Test, ROC y PR

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador | UDLA  
*15 de Abril, 2026*

---

## 0. Tipos de Aprendizaje — ¿Dónde estamos parados?

Machine Learning no es una cosa sola. Hay **paradigmas** según cómo aprende el modelo:

| Paradigma | Cómo aprende | Ejemplo cotidiano |
|---|---|---|
| **Supervisado** | Con ejemplos que ya tienen la respuesta ($X \to y$) | Profesor corrige 100 exámenes |
| **No supervisado** | Sin respuestas. Encuentra estructura por sí solo. | Organizar fotos en álbumes sin categorías |
| **Por refuerzo** | Prueba acciones en un entorno y recibe recompensas | Perro aprende trucos con premios |

### Dentro de supervisado:

- **Regresión**: predecir un número continuo → *costo de seguro (clase 13), precio, temperatura*
- **Clasificación**: predecir una categoría → *renuncia (clase 15), ACV (hoy), spam*
  - **Binaria** (2 clases) → hoy
  - **Multiclase** (3+) → setosa/versicolor/virginica

### Dentro de no supervisado:

- **Clustering** → K-Means, DBSCAN (segmentación de clientes)
- **Reducción de dimensionalidad** → PCA, t-SNE, UMAP

Hoy estamos en **supervisado → clasificación binaria**, pero vamos a dejar de entrenar modelos nuevos por un momento para aprender a **evaluarlos bien**.

---

## 1. Plotly — la tercera librería de visualización

Ya conocemos **Matplotlib** y **Seaborn**. Hoy sumamos **Plotly** — gráficos *interactivos*.

| | Matplotlib | Seaborn | Plotly |
|---|---|---|---|
| Nivel | Bajo | Alto | Alto |
| Salida | Imagen estática | Imagen estática | HTML interactivo |
| Hover | ❌ | ❌ | ✅ |
| Zoom | ❌ | ❌ | ✅ |
| Dashboards web | ❌ | ❌ | ✅ |
| Control total | ✅ | Medio | Medio |

### Regla práctica

- **PDF / paper / reporte** → Seaborn
- **Personalización total / publicación** → Matplotlib
- **Web / dashboard / presentación con interacción** → Plotly

In [ ]:
# Instalar plotly si no lo tienen
# !pip install plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid", context="notebook")
print("Listo.")

### 1.1 Plotly Express — la API concisa

Plotly tiene dos APIs: `plotly.graph_objects` (detallada) y `plotly.express` (rápida, trabaja con DataFrames). Usaremos `express` para casi todo.

La sintaxis es casi igual a Seaborn:

| Seaborn | Plotly Express |
|---|---|
| `sns.scatterplot` | `px.scatter` |
| `sns.histplot` | `px.histogram` |
| `sns.boxplot` | `px.box` |
| `sns.countplot` / `barplot` | `px.bar` |
| `sns.heatmap` | `px.imshow` |
| `hue=` | `color=` |

In [ ]:
# Dataset iconico de Hans Rosling: paises, esperanza de vida, PIB y poblacion (1952-2007)
df_gap = px.data.gapminder()
print(f"{len(df_gap)} filas — {df_gap['country'].nunique()} paises x {df_gap['year'].nunique()} anios")
df_gap.head()

In [ ]:
# Scatter clasico de Rosling: PIB vs esperanza de vida
gap_2007 = df_gap[df_gap["year"] == 2007]
fig = px.scatter(gap_2007, x="gdpPercap", y="lifeExp",
                 color="continent", size="pop",
                 hover_name="country", size_max=60,
                 log_x=True,
                 title="PIB per capita vs Esperanza de vida (2007)")
fig.show()

**Prueben:** pasen el mouse sobre los puntos, hagan zoom arrastrando, doble click para resetear.

In [ ]:
# Histograma: distribucion de esperanza de vida en 2007
fig = px.histogram(gap_2007, x="lifeExp", color="continent",
                   nbins=20, opacity=0.75,
                   title="Distribucion de esperanza de vida por continente (2007)")
fig.show()

In [ ]:
# Box: esperanza de vida por continente
fig = px.box(gap_2007, x="continent", y="lifeExp", color="continent",
             title="Esperanza de vida por continente (2007)")
fig.show()

### 1.2 Personalizar con `update_layout`

En Plotly no hay `plt.title()` ni `plt.figure()`. Todo se configura **en la figura**:

In [ ]:
fig = px.scatter(gap_2007, x="gdpPercap", y="lifeExp",
                 color="continent", size="pop",
                 hover_name="country", log_x=True, size_max=55)

fig.update_layout(
    title="Mundo 2007: PIB vs longevidad (personalizado)",
    xaxis_title="PIB per capita (log)",
    yaxis_title="Esperanza de vida (anios)",
    template="plotly_white",
    width=850, height=500,
    font=dict(size=13),
)
fig.show()

**Templates útiles:** `"plotly"`, `"plotly_white"`, `"plotly_dark"`, `"simple_white"`, `"ggplot2"`, `"seaborn"`.

### 1.3 Gráficos 3D — el superpoder de Plotly

Una cosa que **solo** se puede con Plotly: gráficos 3D que uno **rota con el mouse**.

In [ ]:
df_iris = px.data.iris()

fig = px.scatter_3d(df_iris,
    x="sepal_length", y="sepal_width", z="petal_length",
    color="species", size="petal_width",
    opacity=0.75,
    title="Iris en 3D — rotame con el mouse")
fig.show()

**Click y arrastra** para rotar. **Scroll** para zoom. **Doble click** para resetear.

### 1.4 Otros tipos de gráficos

In [ ]:
# Sunburst: jerarquia continente -> pais, tamano = poblacion
fig = px.sunburst(gap_2007, path=["continent", "country"], values="pop",
                  title="Poblacion mundial: continente -> pais (2007)",
                  color="lifeExp", color_continuous_scale="RdYlGn")
fig.show()

In [ ]:
# Treemap: mismas variables, otra representacion visual
fig = px.treemap(gap_2007, path=["continent", "country"], values="pop",
                 color="lifeExp", color_continuous_scale="RdYlGn",
                 title="Treemap: poblacion mundial coloreada por longevidad")
fig.show()

In [ ]:
# Parallel coordinates: ver patrones en muchas variables
fig = px.parallel_coordinates(df_iris,
    color="species_id",
    dimensions=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    title="Parallel coordinates de Iris")
fig.show()

In [ ]:
# Density heatmap: concentracion de paises en el espacio PIB-longevidad
fig = px.density_heatmap(df_gap, x="gdpPercap", y="lifeExp",
                         nbinsx=25, nbinsy=25, log_x=True,
                         color_continuous_scale="Reds",
                         title="Densidad global (1952-2007): PIB vs longevidad")
fig.show()

### Ejercicio 1: Primer contacto con Plotly (8 min)

Cargaremos el dataset de stroke en la siguiente sección. Mientras tanto, con `df_gap`:

1. Hagan un `px.violin` de `lifeExp` agrupado por `continent`, con `box=True` y `points="all"`.
2. Cambien el `template` a `"plotly_dark"`.
3. **Animación temporal:** rehagan el scatter de Rosling con `animation_frame="year"` usando `df_gap` completo (no filtren por año). Denle play al botón de abajo.
4. **Busquen en la documentación** (plotly.com/python) cómo hacer un `px.choropleth` y prueben mostrar la esperanza de vida de 2007 en un mapa mundial (usen `locations="iso_alpha"`).


In [ ]:
# Tu codigo aqui


---

## 2. Nuevo dataset — Predicción de ACV

Un hospital quiere identificar pacientes con **alto riesgo de ACV** (accidente cerebrovascular) para prevenir antes de que ocurra.

| Campo | Descripción |
|---|---|
| `gender` | Masculino / Femenino |
| `age` | Edad en años |
| `hypertension` | 1 si tiene hipertensión |
| `heart_disease` | 1 si tiene enfermedad cardíaca |
| `ever_married` | Casado alguna vez |
| `work_type` | Tipo de trabajo |
| `Residence_type` | Urbano / Rural |
| `avg_glucose_level` | Glucosa promedio |
| `bmi` | Índice de masa corporal |
| `smoking_status` | Fumador / ex / nunca |
| **`stroke`** | **Target: 1 = tuvo ACV, 0 = no** |

**Asimetría de costos:** un falso negativo (no detectar a alguien en riesgo) puede costar **una vida**. Un falso positivo solo cuesta un chequeo adicional. Esto cambiará cómo evaluamos el modelo.

In [ ]:
URL = ("https://raw.githubusercontent.com/cmosquerat/"
       "arca-diplomado/main/clase-16/stroke.csv")
df = pd.read_csv(URL)

print(f"Shape: {df.shape}")
print(f"Porcentaje con ACV: {df['stroke'].mean()*100:.2f}%")
df.head()

In [ ]:
# Explorar el target con Plotly
fig = px.histogram(df, x="stroke", color="stroke",
                   title="Distribucion del target (desbalance)",
                   color_discrete_sequence=["#2563EB", "#C82B40"])
fig.show()

df["stroke"].value_counts()

**Igual que clase 15:** desbalance fuerte. Si el modelo dijera *"nadie tiene ACV"* acertaría el 95% de las veces — y no serviría para nada.

### Ejercicio 1b: Scatter 3D rotable (8 min)

1. Hagan un `px.scatter_3d` con `x="age"`, `y="avg_glucose_level"`, `z="bmi"`, `color="stroke"`, `opacity=0.6`.
2. Rótenlo con el mouse. ¿Ven algún patrón (zona donde se concentran los positivos)?
3. Cambien `color` por `"hypertension"`.
4. **Bonus:** usen `size="age"` para que los puntos varíen de tamaño.

In [ ]:
# Tu codigo aqui


### Ejercicio 1c: Explora otros tipos (10 min)

Elijan **dos** de estos y pruébenlos sobre `df`:

1. `px.sunburst` con `path=["smoking_status", "gender", "stroke"]` y `values="age"`.
2. `px.treemap` con las mismas variables.
3. `px.parallel_coordinates` con `["age", "avg_glucose_level", "bmi"]`, `color="stroke"`.
4. `px.density_heatmap` de `age` vs `avg_glucose_level`.
5. `px.violin` de `bmi` con `color="stroke"`, `box=True`.

In [ ]:
# Tu codigo aqui


---

## 3. Preparar los datos

In [ ]:
# 1. Eliminar el id (no es feature)
df = df.drop(columns=["id"])

# 2. Revisar NaN
print("NaN por columna:")
print(df.isna().sum())

In [ ]:
# 3. Imputar bmi con la mediana (201 faltantes)
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

# 4. Encoding de categoricas (igual que clase 15)
df_enc = pd.get_dummies(df, drop_first=True, dtype=int)

# 5. Separar X e y
X = df_enc.drop(columns=["stroke"])
y = df_enc["stroke"]

print(f"X: {X.shape}, y: {y.shape}")
X.head(3)

---

## 4. Train/Test Split — evaluar de verdad

### El problema

Imaginen un profesor que da el examen final como *"práctica"* y luego evalúa con **el mismo examen**. Todos sacan 10. Pero ¿aprendieron? No sabemos. Solo **memorizaron**.

Eso es lo que hicimos en clase 15: entrenar y evaluar con los mismos datos. Hoy lo arreglamos.

### La solución

Dividir los datos **antes** de entrenar:
- **Train** (~80%): el modelo aprende con estos.
- **Test** (~20%): el "examen final". El modelo **nunca ve estos datos** durante el entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)   # explicamos abajo

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"% positivos train: {y_train.mean()*100:.2f}%")
print(f"% positivos test:  {y_test.mean()*100:.2f}%")

### ¿Qué hace `stratify=y`?

**Analogía:** imaginen que reparten 100 canicas en dos bolsas (80/20). En la caja original hay 95 azules y 5 rojas.

- **Sin `stratify`:** revuelven todo y agarran 20 al azar → *por mala suerte podrían quedar 0 rojas en el test*. Sin positivos, no pueden medir recall.
- **Con `stratify=y`:** separan por color ANTES y reparten 80/20 **dentro de cada color**. El test queda con su 5% rojo garantizado.

**Regla:** si la clase minoritaria es < 20% del dataset, **siempre** usen `stratify=y`.

Vamos a comprobar la diferencia:

In [ ]:
# Comparar con y sin stratify
print(f"Total original:      {y.mean()*100:.2f}% positivos")
print()

for seed in [1, 2, 3, 4, 5]:
    _, _, _, y_te_sin = train_test_split(X, y, test_size=0.2, random_state=seed)
    _, _, _, y_te_con = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    print(f"seed={seed}:  sin stratify: {y_te_sin.mean()*100:.2f}%  |  con stratify: {y_te_con.mean()*100:.2f}%")

Sin stratify la proporción oscila entre seeds. Con stratify se mantiene casi idéntica al total.

### 4.1 Entrenar y evaluar — honestamente

Cuidado con el **data leakage**: el `StandardScaler` se ajusta solo con train. Si usamos `fit_transform` sobre todo el dataset, el test "ya vio" información que no debería.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit SOLO con train
X_test_s  = scaler.transform(X_test)        # solo transform

modelo = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo.fit(X_train_s, y_train)

y_pred_train = modelo.predict(X_train_s)
y_pred_test  = modelo.predict(X_test_s)

print(f"{'':12}{'TRAIN':>10}{'TEST':>10}")
print(f"{'Accuracy':12}{accuracy_score(y_train, y_pred_train):>10.3f}{accuracy_score(y_test, y_pred_test):>10.3f}")
print(f"{'Precision':12}{precision_score(y_train, y_pred_train):>10.3f}{precision_score(y_test, y_pred_test):>10.3f}")
print(f"{'Recall':12}{recall_score(y_train, y_pred_train):>10.3f}{recall_score(y_test, y_pred_test):>10.3f}")

Las métricas en **test** son las que cuentan. Son parecidas a las de train porque la logística es un modelo simple (no memoriza). En árboles sin podar veríamos la diferencia mucho más grande.

### Ejercicio 2: Tu primer split honesto (10 min)

1. Usen `train_test_split` con `test_size=0.3`, `random_state=7` y `stratify=y`.
2. Verifiquen que la proporción de positivos sea similar en train y test.
3. Escalen con `StandardScaler` (solo fit con train).
4. Entrenen una `LogisticRegression` **sin** `class_weight` y **otra con** `class_weight="balanced"`.
5. Reporten precision y recall de ambos **sobre test**.
6. ¿Cuál tiene mejor recall? ¿A costa de qué?

In [ ]:
# Tu codigo aqui


---

## 5. El umbral — la decisión que estábamos escondiendo

Cuando la regresión logística "predice", realmente responde: *"hay un 67% de probabilidad de que este paciente sufra un ACV"*.

`predict()` convierte esa probabilidad en 0 o 1 usando un **umbral** de **0.5** por defecto. Pero 0.5 es una convención, no una ley.

In [ ]:
# Probabilidades reales del modelo (no etiquetas)
y_proba = modelo.predict_proba(X_test_s)[:, 1]
print("Primeras 10 probabilidades:")
print(np.round(y_proba[:10], 3))

# Lo que predict() hace por dentro:
y_pred_default = (y_proba >= 0.5).astype(int)
print("\nPrimeras 10 predicciones (umbral 0.5):")
print(y_pred_default[:10])

In [ ]:
# Visualizar la distribucion de probabilidades por clase real
fig = px.histogram(
    pd.DataFrame({"proba": y_proba, "real": y_test.map({0: "Sano", 1: "ACV"}).values}),
    x="proba", color="real",
    nbins=40, opacity=0.7, barmode="overlay",
    color_discrete_map={"Sano": "#2563EB", "ACV": "#C82B40"},
    title="Distribucion de probabilidades predichas por clase real")
fig.add_vline(x=0.5, line_dash="dash", line_color="#6B1525",
              annotation_text="umbral = 0.5")
fig.show()

**Observen:** los pacientes reales con ACV (rojo) tienen probabilidades más altas, pero hay solapamiento con los sanos. El umbral de 0.5 es **una línea arbitraria** en medio de ese solapamiento.

In [ ]:
# Probemos otros umbrales
for u in [0.3, 0.5, 0.7]:
    y_pred_u = (y_proba >= u).astype(int)
    p = precision_score(y_test, y_pred_u, zero_division=0)
    r = recall_score(y_test, y_pred_u)
    print(f"Umbral {u}:  precision={p:.3f}  recall={r:.3f}")

Bajar el umbral → más recall (atrapa más ACV) pero menos precision (más falsas alarmas).  
Subir el umbral → más precision pero menos recall.

**El umbral es una decisión de negocio**, no estadística. La pregunta es: *¿cuánto cuesta cada error?*

---

## 6. Curva ROC

En vez de elegir un umbral a ciegas, podemos ver **qué pasa para todos los umbrales posibles** en un solo gráfico.

La **curva ROC** (Receiver Operating Characteristic) muestra:

| Eje | Fórmula | Qué mide |
|---|---|---|
| TPR (y) | VP / (VP + FN) = **Recall** | De los enfermos, ¿cuántos detecto? |
| FPR (x) | FP / (FP + VN) | De los sanos, ¿a cuántos marqué mal? |

- **Umbral bajo** → detecto a todos (TPR ≈ 1) pero también muchos sanos (FPR alto).
- **Umbral alto** → no marco a casi nadie (TPR ≈ 0, FPR ≈ 0).
- **AUC** = área bajo la curva. 1.0 = perfecto, 0.5 = azar.

### 6.1 ¿Cómo se construye la curva ROC? — punto por punto

Antes de usar `roc_curve()` de sklearn, hagámoslo **a mano** con 3 umbrales para entender qué hace.

**Receta:**
1. Fijar un umbral $u$.
2. Convertir probabilidades en predicciones: `y_pred = (y_proba >= u)`.
3. Contar VP, FN, FP, VN en la matriz de confusión.
4. Calcular $\text{TPR} = \frac{VP}{VP+FN}$ y $\text{FPR} = \frac{FP}{FP+VN}$.
5. El par $(FPR, TPR)$ es **UN** punto en la curva. Repetir para muchos umbrales.

In [ ]:
# Calculo manual de 3 puntos en la curva ROC
print(f"{'Umbral':>8} {'VP':>5} {'FN':>5} {'FP':>5} {'VN':>5} {'TPR':>8} {'FPR':>8}")
print("-" * 52)
for u in [0.2, 0.45, 0.70]:
    y_pred_u = (y_proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)
    print(f"{u:>8.2f} {tp:>5d} {fn:>5d} {fp:>5d} {tn:>5d} {tpr:>8.3f} {fpr:>8.3f}")

### 6.2 ¿Cómo se calcula el AUC?

**AUC** = Área Bajo la Curva. **No es "sumar y dividir"**, es una integral numérica.

Sklearn usa la **regla del trapecio**: entre dos puntos consecutivos $(x_i, y_i)$ y $(x_{i+1}, y_{i+1})$ calcula el área del trapecio:

$$\text{área}_i = (x_{i+1} - x_i) \cdot \frac{y_i + y_{i+1}}{2}$$

El AUC es la suma de todas esas áreas: $\text{AUC} = \sum_i \text{área}_i$.

**Interpretación probabilística:** el AUC es la probabilidad de que el modelo asigne un score mayor a un positivo aleatorio que a un negativo aleatorio. AUC = 0.84 significa que el modelo ordena bien los casos el 84% de las veces.

In [ ]:
# Calcular el AUC a mano (regla del trapecio) y comparar con sklearn
fpr, tpr, _ = roc_curve(y_test, y_proba)

auc_manual = 0.0
for i in range(len(fpr) - 1):
    ancho = fpr[i+1] - fpr[i]
    alto_medio = (tpr[i] + tpr[i+1]) / 2
    auc_manual += ancho * alto_medio

from sklearn.metrics import auc as sk_auc
print(f"AUC manual (trapecios): {auc_manual:.4f}")
print(f"AUC sklearn:            {sk_auc(fpr, tpr):.4f}")
print(f"roc_auc_score:          {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, umbrales_roc = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig = px.area(x=fpr, y=tpr,
    title=f"Curva ROC (AUC = {auc:.3f})",
    labels={"x": "FPR (1 - Especificidad)", "y": "TPR (Recall)"},
    width=700, height=500)
fig.add_shape(type="line", line=dict(dash="dash", color="gray"),
              x0=0, x1=1, y0=0, y1=1)
fig.update_layout(template="plotly_white")
fig.show()

Pasen el mouse sobre la curva — Plotly muestra los valores exactos de cada punto. Cuanto más se pega la curva a la esquina superior izquierda, mejor el modelo.

---

## 7. Curva Precision-Recall

### ¿Por qué PR > ROC en datos desbalanceados?

La curva ROC puede verse "bonita" aunque el modelo sea malo — si el desbalance es fuerte. El FPR tiene un denominador enorme (todos los sanos). 100 falsos positivos sobre 4861 sanos = FPR de 2%. **Parece bueno.** Pero si solo hay 249 positivos reales y detectamos 50, la **precisión** es 50/(50+100) = 33%. **Es malo.**

La curva **Precision-Recall** se centra solo en la clase positiva (la que nos importa).

### 7.1 ¿Cómo se construye la curva PR? — punto por punto

Misma receta que ROC, **cambia solo el eje Y**: en vez de FPR usamos Precision.

1. Fijar umbral $u$.
2. `y_pred = (y_proba >= u)`.
3. Contar VP, FN, FP, VN.
4. Calcular $\text{Precision} = \frac{VP}{VP+FP}$ y $\text{Recall} = \frac{VP}{VP+FN}$.
5. El par $(\text{Recall}, \text{Precision})$ es UN punto. Repetir.

In [ ]:
# Calculo manual de 3 puntos en la curva PR
print(f"{'Umbral':>8} {'VP':>5} {'FN':>5} {'FP':>5} {'Precision':>12} {'Recall':>8}")
print("-" * 55)
for u in [0.2, 0.45, 0.70]:
    y_pred_u = (y_proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    recall    = tp / (tp + fn)
    print(f"{u:>8.2f} {tp:>5d} {fn:>5d} {fp:>5d} {precision:>12.3f} {recall:>8.3f}")

### 7.2 ¿Cómo se calcula el AP (Average Precision)?

**AP** = área bajo la curva PR, pero sklearn lo calcula **distinto** al AUC.

En vez de trapecios, usa **rectángulos** — un promedio ponderado:

$$\text{AP} = \sum_n (R_n - R_{n-1}) \cdot P_n$$

Donde $R_n$ y $P_n$ son recall y precision en el n-ésimo umbral. Esto evita sobrestimar la curva, que suele ser *escalonada* en PR (la precisión no siempre decrece monotónicamente).

Un modelo **aleatorio** tiene AP = prevalencia de la clase positiva. En stroke, AP aleatorio ≈ 0.05. Si nuestro modelo tiene AP = 0.27, es **≈ 5× mejor que el azar**.

In [ ]:
# Calcular el AP a mano (rectangulos) y comparar con sklearn
prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_proba)

# sklearn ordena recall DESCendente; lo invertimos para sumar rectangulos correctamente
ap_manual = 0.0
for n in range(1, len(rec_arr)):
    delta_recall = rec_arr[n-1] - rec_arr[n]   # positivo porque va de 1 -> 0
    ap_manual += delta_recall * prec_arr[n-1]

print(f"AP manual (rectangulos):     {ap_manual:.4f}")
print(f"AP sklearn (average_precision_score): {average_precision_score(y_test, y_proba):.4f}")
print(f"\nPrevalencia (baseline aleatorio):    {y_test.mean():.4f}")
print(f"Mejora sobre azar: {ap_manual / y_test.mean():.1f}x")

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

prec, rec, umbrales_pr = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

fig = px.area(x=rec, y=prec,
    title=f"Curva Precision-Recall (AP = {ap:.3f})",
    labels={"x": "Recall", "y": "Precision"},
    width=700, height=500)
fig.add_hline(y=y_test.mean(), line_dash="dash", line_color="gray",
              annotation_text=f"baseline = {y_test.mean():.3f} (% positivos)")
fig.update_layout(template="plotly_white")
fig.show()

**Lectura:**
- **AP** (Average Precision) = área bajo la curva PR.
- En datos muy desbalanceados, un AP de 0.30 ya es *notable* (baseline es ~0.05).
- Un modelo aleatorio se quedaría sobre la línea horizontal punteada.

---

## 8. Umbral óptimo — lógica de negocio

### Tres estrategias comunes

1. **F1 máximo**: balance automático entre precision y recall.
2. **Recall mínimo**: "detecta al menos 80% de los ACV" → busca el umbral más alto que cumpla.
3. **Costo esperado**: multiplicar errores por su costo y minimizar.

Vamos por la tercera — la más honesta para un problema clínico.

In [ ]:
# Barrido de umbrales: precision y recall
umbrales = np.arange(0.05, 0.95, 0.02)
registros = []
for u in umbrales:
    y_pred_u = (y_proba >= u).astype(int)
    registros.append({
        "umbral": u,
        "precision": precision_score(y_test, y_pred_u, zero_division=0),
        "recall":    recall_score(y_test, y_pred_u),
    })
df_u = pd.DataFrame(registros)

fig = px.line(df_u.melt(id_vars="umbral", var_name="metrica", value_name="valor"),
              x="umbral", y="valor", color="metrica",
              title="Precision y Recall vs Umbral",
              color_discrete_map={"precision": "#2563EB", "recall": "#C82B40"},
              width=750, height=450)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Caso de negocio: costo asimetrico
COSTO_FN = 10_000   # no detectar un ACV (tratamiento tardio)
COSTO_FP = 200      # falsa alarma (chequeo adicional)

costos = []
for u in umbrales:
    y_pred_u = (y_proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    costo_total = fn * COSTO_FN + fp * COSTO_FP
    costos.append({"umbral": u, "costo": costo_total, "FN": fn, "FP": fp})

df_c = pd.DataFrame(costos)
idx_opt = df_c["costo"].idxmin()
u_opt = df_c.loc[idx_opt, "umbral"]

print(f"Umbral optimo: {u_opt:.2f}")
print(f"Costo minimo:  ${df_c.loc[idx_opt, 'costo']:,}")
print(f"  FN: {df_c.loc[idx_opt, 'FN']}  (no detectados)")
print(f"  FP: {df_c.loc[idx_opt, 'FP']}  (falsas alarmas)")

fig = px.line(df_c, x="umbral", y="costo",
              title=f"Costo esperado vs umbral (optimo: {u_opt:.2f})",
              width=750, height=400)
fig.add_vline(x=u_opt, line_dash="dash", line_color="#C82B40")
fig.update_layout(template="plotly_white")
fig.show()

El umbral óptimo **no es 0.5**. Cuando el costo de un FN es 50x el de un FP, el modelo debe ser mucho más sensible.

---

### Ejercicio 3: Tu curva ROC y PR (12 min)

Con el modelo de clase 15 (sin balancear) aplicado sobre este train/test:

1. Entrenen una `LogisticRegression(max_iter=1000)` **sin** `class_weight`.
2. Obtengan `y_proba = modelo.predict_proba(X_test_s)[:, 1]`.
3. Calculen y grafiquen la **curva ROC** con Plotly. Reporten AUC.
4. Calculen y grafiquen la **curva PR**. Reporten AP.
5. Prueben 5 umbrales distintos (0.1, 0.3, 0.5, 0.7, 0.9) y reporten precision, recall y matriz de confusión.
6. ¿Qué umbral eligirían si un FN cuesta 50x más que un FP?

In [ ]:
# Tu codigo aqui


### Ejercicio 4 (retador): Comparar dos modelos en la misma curva (10 min)

1. Entrenen dos modelos sobre el mismo train:
   - `modelo_a = LogisticRegression(max_iter=1000)`  # sin balancear
   - `modelo_b = LogisticRegression(max_iter=1000, class_weight="balanced")`
2. Obtengan las probabilidades de ambos.
3. Grafiquen **ambas curvas ROC en el mismo gráfico**. Tip: usen `fig.add_trace(go.Scatter(x=..., y=..., name=...))`.
4. Hagan lo mismo con las curvas PR.
5. **Pregunta:** ¿cuál tiene mejor AUC? ¿Y mejor AP? ¿El desbalance afecta más al AUC o al AP?

In [ ]:
# Tu codigo aqui


---

## Resumen

### Visualización

| Concepto | Código clave |
|---|---|
| Plotly Express | `px.scatter`, `px.histogram`, `px.box`, `px.violin` |
| 3D interactivo | `px.scatter_3d` — rotable con el mouse |
| Jerárquicos | `px.sunburst`, `px.treemap` |
| Multivariados | `px.parallel_coordinates` |
| Personalizar | `fig.update_layout(template=..., width=..., height=...)` |

### Evaluación rigurosa

| Concepto | Código clave |
|---|---|
| Partir datos | `train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)` |
| Evitar leakage | `scaler.fit_transform(X_train)` + `scaler.transform(X_test)` |
| Probabilidades | `modelo.predict_proba(X)[:, 1]` |
| Curva ROC | `roc_curve`, `roc_auc_score` |
| Curva PR | `precision_recall_curve`, `average_precision_score` |
| Umbral por costo | minimizar `FN * c_FN + FP * c_FP` sobre un barrido |

### Cuándo usar qué

- **AUC (ROC)**: dataset razonablemente balanceado, quiero una métrica general.
- **AP (PR)**: dataset desbalanceado — es la métrica correcta.
- **Recall**: cuando el FN duele mucho (medicina, fraude).
- **Precision**: cuando el FP duele mucho (marcar a un cliente como moroso por error).
- **F1**: no sé cuál pesa más, quiero un balance.

**Próxima clase:** árboles de decisión — un modelo nuevo que hereda todo este instrumental de evaluación.